[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03b_morbo.ipynb)

# 03b — Multi-Objective Bayesian Optimization with MORBO

## 1. Overview

This notebook runs **MORBO**, the proposed method for multi-band antenna tilt coordination, and reads it against the
baseline of notebook 03a.

**Question.** Does a model-guided multi-objective search find better trade-offs between coverage, separation and
throughput than random search, for the same number of ray-traced evaluations?

**Goals.**

- Run MORBO on the same simulator, scenario, objectives and evaluation budget as random search.
- Show how its trust region spends that budget and what front it finds.
- Measure what its chosen configuration changes on every KPI, beside random search's.

The problem formulation is described in notebook 01, the setup tables and the objectives in notebook 03a
(sections 3 and 7); they are not repeated here.

**Outputs.** One run directory under `outputs/optim/morbo/`, tables under `reports/tables/03b_morbo/`,
figures under `reports/figures/03b_morbo/`. **Requirements.** A CUDA GPU, and notebook 03a run first. The shell
equivalent of the run is `task bo`.


In [ ]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna.rt", "sionna-rt"), ("botorch", "botorch")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import load_config
from src.core.sector import read_sectors, site_frame
from src.evaluation import compare, plots
from src.evaluation import runs as run_store
from src.evaluation.export import readable, save_table
from src.optim.methods.morbo.search import MORBO, RESTART, failure_tolerance, local_size
from src.optim.objective import OBJECTIVE_NAMES, hypervolume_contributions, pareto_mask
from src.optim.run import run
from src.optim.space import TiltSpace
from src.simulation.radio import read_manifest
from src.utils.plotting import label, save_fig, setup_plotting

cfg = load_config(overrides=CONFIG_OVERRIDES)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path(cfg.reports.figures_dir) / "03b_morbo")
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path(cfg.reports.tables_dir) / "03b_morbo")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

METHODS = ("morbo", "random")


## 2. Method

MORBO (Daulton et al., UAI 2022, `optim/method=morbo`) keeps one or more trust regions, boxes in the
unit cube the tilt box is rescaled to, and searches inside them for points that add hypervolume:

1. Evaluate the current configuration, then an initial Sobol design of `n_init` points, the same design random search
   starts with.
2. Centre each region on the Pareto point with the largest hypervolume contribution it may take: inside the region
   if one is free, else anywhere. Another region's centre and a collapsed region's old centre are taken.
3. Fit one GP per objective (constant mean, Matérn-5/2 kernel with ARD under BoTorch's dimension-scaled lengthscale
   prior, standardized outcome) to every evaluation
   within a cube of side $2L$ around the centre, topped up with the nearest points to at least $\min(250, 2d)$, the
   inputs rescaled from that cube. Evaluations are shared: a region's model sees what any region found.
4. For each point of the batch, draw `n_candidates` fresh candidates per region by perturbing random Pareto points
   inside it, each dimension with probability $p_n = p_0\bigl(1 - 0.5\log(n - n_0 + 1) / \log(N - n_0 + 1)\bigr)$,
   $p_0 = \min(k/d, 1)$, $k$ = `perturbed_dimensions`.
5. Pick a batch of `batch_size` one point at a time: draw a joint posterior sample over each region's candidates and
   the points already picked inside it, and take the candidate whose sample adds the most hypervolume to the observed
   front and those picked points (BoTorch's `FastNondominatedPartitioning` turns that into one vectorized computation
   per draw). When no candidate adds any, a random scalarisation of the objectives breaks the tie.
6. Ray-trace the batch. A region's points succeed when they raise the hypervolume by the share `improvement`;
   after $\max(10, \lceil d/3 \rceil)$ failed evaluations in a row its side halves. Regions never grow. Below
   `length_min` a region restarts, at a side that decays with the budget used, around the point a random hypervolume
   scalarisation ranks best in one draw of a global GP over the initial design and earlier restart points.

Every proposal is snapped to the tilt resolution before it is traced. Every candidate is ray-traced and scored exactly
as in 03a; the GPs only choose where to look, and no reported number comes from them.

## 3. Configuration

In [ ]:
space = TiltSpace.from_config(cfg)

morbo_cfg = load_config(overrides=["optim/method=morbo", *CONFIG_OVERRIDES])
method = morbo_cfg.optim.method
region = method.trust_region
settings = pd.DataFrame(
    {
        "Parameter": [
            "Optimized variables",
            "Tilt resolution [°]",
            "Evaluations (excluding the incumbent)",
            "Budget split",
            "Batch size",
            "Trust regions",
            "Trust region length: initial, minimum",
            "Failure tolerance (evaluations)",
            "Improvement for a success (share of the hypervolume)",
            "Perturbed dimensions per candidate at the start (expected)",
            "Candidates per region and round",
            "Fewest points per local model",
            "Kernel",
            "Hypervolume reference point",
            "Search seed",
        ],
        "Value": [
            space.n_dim,
            f"{space.resolution_deg:g}",
            int(method.budget.n_init) + int(method.budget.n_iter),
            f"n_init={method.budget.n_init}, n_iter={method.budget.n_iter}",
            int(method.budget.batch_size),
            int(region.count),
            f"{float(region.length_init):g}, {float(region.length_min):g}",
            failure_tolerance(space.n_dim),
            f"{float(region.improvement):g}",
            f"{min(float(region.perturbed_dimensions), space.n_dim):g} of {space.n_dim}",
            int(method.n_candidates),
            local_size(space.n_dim),
            "Matérn-5/2 with ARD, one GP per objective",
            "origin (0, 0)",
            int(cfg.optim.seed),
        ],
    }
)
save_table(settings, "morbo_configuration")
settings


## 4. Runs

A MORBO run already on disk for the current scenario is reused; delete its directory under `outputs/optim/morbo/` to
repeat it. Runs from an earlier scenario are ignored. The baseline run is read from disk, not repeated, and every run
is checked comparable with the current config.

In [ ]:
scenario = str(read_manifest(cfg)["scenario_id"])


def on_disk():
    # The newest finished run of each method on the current scenario.
    current = [r for r in run_store.discover(cfg.optim.output.dir) if r.scenario_id == scenario]
    return run_store.latest_per_method(current)


if "morbo" not in {r.method for r in on_disk()}:
    run(load_config(overrides=["optim/method=morbo", *CONFIG_OVERRIDES]))

runs = [r for r in on_disk() if r.method in METHODS]
missing = [m for m in METHODS if not any(r.method == m for r in runs)]
if missing:
    raise run_store.RunError(
        f"No {', '.join(missing)} run on scenario {scenario}; run notebook 03a first."
    )
checks = run_store.verify(runs, cfg, scenario)
run_store.require(checks)
print(f"{len(runs)} runs; {int(checks['holds'].sum())} of {len(checks)} comparability checks hold")
morbo = next(r for r in runs if r.method == "morbo")

# Each method's largest hypervolume contribution on the evaluation KPIs among the
# configurations it proposed, as notebook 04 picks it.
searched = compare.candidates(runs)
picks = {m: compare.method_pick(searched, m) for m in METHODS}

## 5. Search Progress

Hypervolume of every evaluation so far, for MORBO and random search, on the search objectives and on notebook 04's
three KPIs, then how much each search added beyond its initial design.

In [ ]:
figure = plots.convergence_plot(compare.convergence(runs))
save_fig(figure, "search_progress")
plt.show()

volumes = readable(compare.hypervolume_table(runs))
save_table(volumes, "hypervolume")
volumes


### Where MORBO spent its budget

Every evaluation of MORBO's run on each objective, marked by what proposed it: the Sobol design, a trust-region
proposal, or a region's restart point.


In [ ]:
history = morbo.history
proposer = history["generation_node"].replace({"attached": "Current configuration"})
colours = {
    "Current configuration": plots.COLOURS["incumbent"],
    "Sobol": plots.COLOURS["random"],
    MORBO: plots.COLOURS["morbo"],
    RESTART: "tab:purple",
}
values = history[list(OBJECTIVE_NAMES)].to_numpy(float)
on_front = pareto_mask(values)
restarts = int(((history["phase"] == "init") & (history["generation_node"] == RESTART)).sum())

figure, axes = plt.subplots(
    1, len(OBJECTIVE_NAMES), figsize=(5.4 * len(OBJECTIVE_NAMES), 4.6), constrained_layout=True
)
for axis, name in zip(axes, OBJECTIVE_NAMES, strict=True):
    for key, group in history.groupby(proposer, sort=False):
        axis.scatter(group["iteration"], group[name], s=14, color=colours[key], label=key)
    axis.scatter(
        history.loc[on_front, "iteration"], history.loc[on_front, name], s=60,
        facecolors="none", edgecolors="black", label="Pareto front",
    )
    axis.set(xlabel="Evaluations", ylabel=f"{label(name)} (maximise)")
axes[0].legend(fontsize=8)
figure.suptitle(f"MORBO: every evaluation, {restarts} restarts")
save_fig(figure, "morbo_evaluations")
plt.show()

by_proposer = history.assign(pareto=on_front).groupby(proposer, sort=False).agg(
    Evaluations=("iteration", "size"),
    **{f"Mean {label(name).lower()}": (name, "mean") for name in OBJECTIVE_NAMES},
    **{"On the Pareto front": ("pareto", "sum")},
).rename_axis("Proposed by")
save_table(by_proposer.reset_index(), "morbo_evaluations_by_proposer")
by_proposer


**Observations.** Both searches share the 8-point Sobol design, which takes the hypervolume on the objectives from
today's 2.540 to 2.662. MORBO passes random search's final value (2.765) at evaluation 36 and is still adding to it at
evaluation 72, ending at 2.857; on the three KPIs it ends at 52.54 against random search's 47.05.

**The model is steering.** MORBO's 64 proposals average coverage 0.9363, separation 0.6852 and throughput 4.2199,
against 0.9329, 0.6739 and 4.0751 for the Sobol design. 51 of its candidates beat today on all three objectives,
against 10 of random search's. All three points of its front on the objectives are trust-region proposals. No
region restarted, so the trust region never collapsed.

## 6. Pareto Front

Every configuration both searches evaluated, on each pair of notebook 04's three KPIs; the stars are each method's
chosen configuration, its largest hypervolume contribution on the three among the points it proposed. Then MORBO's
own front on the three KPIs, with each point's hypervolume contribution.

In [ ]:
kpis = compare.EVALUATION_KPIS
for x, y in ((kpis[0], kpis[1]), (kpis[0], kpis[2]), (kpis[1], kpis[2])):
    figure = plots.tradeoff_scatter(searched, x, y, picks)
    save_fig(figure, f"tradeoff_{x}_vs_{y}")
    plt.show()

# MORBO's own front on the three KPIs, ranked by hypervolume contribution.
mine = compare.with_rates(history)
kpi_values = mine[list(kpis)].to_numpy(float)
kpi_front = pareto_mask(kpi_values)
front = mine.loc[kpi_front, ["iteration", "phase", "generation_node", *kpis]].assign(
    hv_contribution=hypervolume_contributions(kpi_values)[kpi_front],
).sort_values("hv_contribution", ascending=False)
front = readable(front)
save_table(front, "morbo_pareto_front")
front

**Observations.** MORBO's front on the three KPIs holds eight points: today's configuration, which alone holds the
top separation rate (0.6563), and seven trust-region proposals. Its chosen point is **evaluation 71**: coverage rate
0.9416, separation rate 0.6237 and median throughput 84.9 Mbit/s, against today's 0.9326, 0.6563 and 60.3. It is also
the largest hypervolume contribution on the search objectives (coverage 0.9416, separation 0.6997, throughput 4.3184,
against today's 0.9326, 0.6800 and 4.0051).

No candidate of either search raises the separation rate above today's: every one trades it for coverage and
throughput. The whole front, with its tilts, is published by notebook 04 (`reports/outputs/pareto_tilts.csv`).

## 7. Tilt Configuration of the Chosen Point

Current and proposed tilt per sector-band of MORBO's chosen point. Negative change is uptilt.

In [ ]:
chosen = searched.loc[picks["morbo"]]
tilt = readable(compare.tilt_table(chosen, space))
save_table(tilt, "tilt_morbo_pick")
print(f"{label('morbo')}: run {morbo.run_id}, evaluation {int(chosen['iteration'])}")
display(tilt)
figure = plots.tilt_delta_heatmap(compare.tilt_table(chosen, space), "morbo")
save_fig(figure, "tilt_delta_heatmap_morbo")
plt.show()

**Observations.** The chosen point moves 44 of the 45 sector-bands, 31 up and 13 down, to between 0.0° and 19.6°.
Net it uptilts every band: 2600 MHz by 4.2°, 1800 MHz by 3.7° and 700 MHz by 3.0°. The largest single change is
12.0° (n4s2's 1800 MHz carrier, to the 0° bound).

## 8. Final Results

### KPI comparison

The network KPIs of the current configuration and of each method's chosen configuration, on every UE, then the
budget each run spent and what it cost.

In [ ]:
incumbent = morbo.incumbent_kpi
comparison = compare.delta_table(incumbent, compare.kpi_vector(searched.loc[picks["morbo"]]))
random_pick = compare.kpi_vector(searched.loc[picks["random"]])
comparison = readable(
    comparison.assign(random=[getattr(random_pick, name) for name in comparison["kpi"]])
).rename(columns={"before": "Current configuration", "after": label("morbo")})
save_table(comparison, "kpi_comparison")
display(comparison)

budget = readable(compare.search_budget(runs))
save_table(budget, "search_budget")
budget

**Observations.** Against today, MORBO's chosen point improves six of the seven network KPIs and worsens one:
- the hole rate falls from 0.0674 to 0.0584 and the weak rate from 0.3269 to 0.2262;
- overlap neighbours per covered tile fall from 0.992 to 0.881, but the band-collapsed co-band overlap rate rises
  from 0.3437 to 0.3763;
- the 5th-percentile throughput rises from 8.2 to 12.9 Mbit/s, the median from 60.3 to 84.9 and the mean from 88.7
  to 111.7.

Random search's chosen point, evaluation 2 of the shared initial design, reaches a hole rate of 0.0671 and an overlap
rate of 0.3839.

Both runs spent 73 evaluations the same way. MORBO took 5.6 minutes of wall clock, of which 3.2 were ray tracing
(2.6 s per evaluation); GP fitting and acquisition make up the rest. Random search took 4.4 minutes, 3.9 of them ray
tracing (3.2 s per evaluation).

## 9. Key Observations

- **MORBO searches better at the same budget.** Its final hypervolume is 2.857 against 2.765 on the objectives and
  52.54 against 47.05 on the three KPIs, and it passes random search's final value at evaluation 36 of 73.
- **The chosen point raises coverage and capacity.** Coverage rate 0.9326 to 0.9416, median throughput 60.3 to
  84.9 Mbit/s, and six of the seven network KPIs improve.
- **Separation does not improve band-collapsed.** No candidate beats today's separation rate, although the chosen
  point raises every band's own separation rate (notebook 04, Section 8).
- **The chosen point uptilts every band on net,** 2600 MHz most, and moves load onto 1800 MHz.

## 10. Limitations

- **The pick depends on the reference point and on the candidate set.** Against the origin and with the incumbent
  in the set, today's configuration has the largest contribution on the three KPIs, so the chosen point is taken from
  MORBO's own proposals.
- **Budget.** 73 evaluations in 45 dimensions; MORBO's hypervolume was still rising at evaluation 72, and no restart
  was triggered, so the restart was not exercised on this run.
- **One trust region.** `trust_region.count` is 1; the paper's default of 5 was not tried.
- **GP prior.** The local GPs carry BoTorch's dimension-scaled lengthscale prior, which the paper does not use; without
  it the first ray-traced run failed to fit.
- **The objectives can disagree with the KPIs.** The soft separation objective rose while the band-collapsed overlap
  rate rose too: one is a per-tile product over bands, the other counts any crowded band.
- **Winner's curse.** The chosen point is the best of 72 evaluated points, so its scores are biased upward.
- **No movement constraint.** Tilt change is reported, not penalised, so the chosen point moves 44 of 45 sector-band
  tilts, one by 12.0°.

## 11. Conclusion

MORBO is a working model-guided multi-objective search for multi-band tilt on this scenario. It:

- reaches a larger hypervolume than random search at the same budget and seed, from the same Sobol design, on the
  objectives it searched and on notebook 04's three KPIs;
- finds configurations that raise coverage and median throughput well beyond random search's;
- does not find a configuration that also beats today's band-collapsed separation rate.

Notebook 04 tests whether MORBO's candidates are better than random search's and maps where the chosen point gains.